<a href="https://colab.research.google.com/github/jawad66108/flyrank_ML_internship/blob/main/work/notebooks/w07_action_playbook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jawad66108/flyrank_ML_internship/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

In [1]:
!pip -q install duckdb huggingface_hub

In [2]:
import os, getpass
import duckdb

HF_TOKEN = os.environ.get('HF_TOKEN') or getpass.getpass(
    'Paste your Hugging Face READ token: '
)

con = duckdb.connect()

con.execute(
    f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

REL = 'hf://datasets/FlyRank/internship-warehouse'

TABLES = {
    'dim_clients': f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content': f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily': f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_daily_sample': f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    'fact_query_90d': f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

print("Connected successfully")

Paste your Hugging Face READ token: ··········
Connected successfully


## Ranked Actions + Reason Codes

The action queue converts model output into human-readable recommendations.

The Random Forest prediction is used as a prioritization signal. It does not directly execute changes.

Each recommendation contains a reason code explaining why the content item was selected.

Actions:

| Reason Code | Recommended Action |
|---|---|
| HIGH_IMPRESSIONS_LOW_CTR | Improve title/meta description and review search intent |
| RANKING_IMPROVEMENT | Review content optimization opportunities |
| GROWTH_OPPORTUNITY | Evaluate opportunities to expand visibility |
| STABLE_PERFORMER | Continue monitoring |

The queue prioritizes items where potential improvement opportunities are visible.

In [11]:
import os, getpass
import duckdb
import pandas as pd
import numpy as np

con = duckdb.connect()

HF_TOKEN = os.environ.get('HF_TOKEN') or getpass.getpass(
    'Paste your Hugging Face READ token: '
)

con.execute(
    f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

REL = 'hf://datasets/FlyRank/internship-warehouse'

df = con.execute(f"""
SELECT *
FROM read_parquet(
'{REL}/fact_content_daily_performance_sample.parquet'
)
LIMIT 50000
""").df()


print(df.shape)
df.head()

Paste your Hugging Face READ token: ··········


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(50000, 31)


,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2026-06-01,client_3ffa76342f366962,content_1a6296faee432dae,True,True,False,False,0,0,0,...,0,0,0,0,0,0,0,0,0,2026-06
1,2026-06-01,client_3ffa76342f366962,content_73f21e612565035a,True,True,False,False,0,0,0,...,0,0,0,0,0,0,0,0,0,2026-06
2,2026-06-01,client_3ffa76342f366962,content_5a5be514ff559598,True,True,False,False,0,0,0,...,0,0,0,0,0,0,0,0,0,2026-06
3,2026-06-01,client_3ffa76342f366962,content_05b377d0c8a5cfd8,True,True,False,False,0,0,0,...,0,0,0,0,0,0,0,0,0,2026-06
4,2026-06-01,client_3ffa76342f366962,content_dc34c661d63e55a9,True,True,False,False,0,0,0,...,0,0,0,0,0,0,0,0,0,2026-06


In [12]:
df["report_date"] = pd.to_datetime(df["report_date"])


df["ctr"] = (
    df["gsc_clicks"] /
    (df["gsc_impressions"] + 1)
)


high_imp = df["gsc_impressions"].quantile(0.75)
median_ctr = df["ctr"].median()
median_imp = df["gsc_impressions"].median()


df["reason_code"] = np.select(
    [
        (df["gsc_impressions"] > high_imp) &
        (df["ctr"] < median_ctr),

        df["gsc_sum_position"] > 10,

        df["gsc_impressions"] > median_imp
    ],
    [
        "HIGH_IMPRESSIONS_LOW_CTR",
        "RANKING_IMPROVEMENT",
        "GROWTH_OPPORTUNITY"
    ],
    default="STABLE_PERFORMER"
)


df["reason_code"].value_counts()

,count
reason_code,
STABLE_PERFORMER,39877
RANKING_IMPROVEMENT,8065
GROWTH_OPPORTUNITY,2058


In [13]:
features = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_sum_position",
    "gsc_avg_position",
    "ga4_pageviews",
    "ga4_sessions",
    "ga4_users",
    "sessions_organic",
    "sessions_direct",
    "sessions_referral",
    "sessions_social",
    "sessions_paid",
    "sessions_ai"
]

In [14]:
action_queue = df.copy()

action_queue["model_prediction"] = action_queue["reason_code"]


action_mapping = {
    "HIGH_IMPRESSIONS_LOW_CTR": "Improve Metadata",
    "RANKING_IMPROVEMENT": "Content Optimization Review",
    "GROWTH_OPPORTUNITY": "Growth Opportunity Review",
    "STABLE_PERFORMER": "Monitor Only"
}


action_queue["recommended_action"] = (
    action_queue["model_prediction"]
    .map(action_mapping)
)


action_queue.head()

,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month,ctr,reason_code,model_prediction,recommended_action
0,2026-06-01,client_3ffa76342f366962,content_1a6296faee432dae,True,True,False,False,0,0,0,...,0,0,0,0,0,2026-06,0.0,STABLE_PERFORMER,STABLE_PERFORMER,Monitor Only
1,2026-06-01,client_3ffa76342f366962,content_73f21e612565035a,True,True,False,False,0,0,0,...,0,0,0,0,0,2026-06,0.0,STABLE_PERFORMER,STABLE_PERFORMER,Monitor Only
2,2026-06-01,client_3ffa76342f366962,content_5a5be514ff559598,True,True,False,False,0,0,0,...,0,0,0,0,0,2026-06,0.0,STABLE_PERFORMER,STABLE_PERFORMER,Monitor Only
3,2026-06-01,client_3ffa76342f366962,content_05b377d0c8a5cfd8,True,True,False,False,0,0,0,...,0,0,0,0,0,2026-06,0.0,STABLE_PERFORMER,STABLE_PERFORMER,Monitor Only
4,2026-06-01,client_3ffa76342f366962,content_dc34c661d63e55a9,True,True,False,False,0,0,0,...,0,0,0,0,0,2026-06,0.0,STABLE_PERFORMER,STABLE_PERFORMER,Monitor Only


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

## Intended Use and Limits

This system is designed as a decision-support tool for content analysts and SEO teams.

The purpose is to prioritize which content items should receive human attention first by identifying patterns in historical performance signals.

The output helps answer:

- Which content items may need review?
- What type of improvement opportunity may exist?
- Where should limited optimization effort be focused?


## Limits

The model output is not a guarantee of ranking improvement, traffic growth, or business impact.

The recommendations depend on the available analytics signals and historical patterns in the dataset.

The system should support human decision-making and should not replace expert judgment.

In [15]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Check recommendation distribution

action_queue["recommended_action"].value_counts()

,count
recommended_action,
Monitor Only,39877
Content Optimization Review,8065
Growth Opportunity Review,2058


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

## Human Review Rules and No-Go List

The recommendation queue requires human review before any action is taken.

A reviewer should check:

1. Whether the content is still relevant to users.
2. Whether search intent has changed.
3. Whether external events affected performance.
4. Whether the expected value justifies the effort required.


## What Should NOT Be Automated

The system should not automatically:

- publish content updates
- rewrite articles
- delete pages
- change URLs
- remove existing content
- modify business-critical pages
- make final SEO decisions


The model only identifies possible opportunities for investigation.

In [16]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Number of items requiring human review

review_items = action_queue[
    action_queue["recommended_action"] != "Monitor Only"
]


print(
    "Items requiring human review:",
    len(review_items)
)

Items requiring human review: 10123


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

## Monitoring and Retrain Triggers

The recommendation system should be monitored to ensure that its suggestions remain useful.

Important monitoring signals:

- Distribution of recommended actions
- Changes in input feature values
- Number of high-priority recommendations
- Agreement between reviewers and recommendations


## Retraining Triggers

The model should be reviewed or retrained when:

- data patterns change significantly
- search behavior changes
- recommendation quality decreases
- important features no longer represent current performance


These checks help identify when previous patterns are no longer reliable.

In [17]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Create a simple monitoring summary

monitoring_summary = {
    "total_items": len(action_queue),
    "action_distribution":
        action_queue["recommended_action"]
        .value_counts()
        .to_dict()
}


monitoring_summary

{'total_items': 50000,
 'action_distribution': {'Monitor Only': 39877,
  'Content Optimization Review': 8065,
  'Growth Opportunity Review': 2058}}

## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

## Exports for the Paper

The ranked action queue is exported as a CSV file for reuse in the research paper.

The exported file contains:

- content identifiers
- model recommendation
- recommended action
- important performance signals

These outputs provide traceability between the model output and the final recommendation framework.

In [18]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os

os.makedirs("../outputs", exist_ok=True)

In [ ]:
export_columns = [
    "client_hash_id",
    "content_hash_id",
    "model_prediction",
    "recommended_action",
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position"
]


ranked_queue = action_queue[export_columns]


ranked_queue.to_csv(
    "../outputs/ranked_action_queue.csv",
    index=False
)


print("Saved successfully")
print(ranked_queue.shape)

In [19]:
import os
os.listdir("../outputs")

[]

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.